# nvMolKit + Nemotron

This notebook demonstrates a guided chemistry agent using NVIDIA Nemotron to call a small set of allow-listed scientific tools backed by nvMolKit on an NVIDIA GPU. Nemotron first reads the BioNeMo Agent Toolkit skill for nvMolKit, learning the library's supported operations, API boundaries, and GPU requirements. It then works through a molecular library one analysis at a time: validating the sample, generating Morgan fingerprints, measuring all-pairs Tanimoto similarity, identifying structural clusters, and generating and minimizing representative conformers.

Each stage follows the same transparent pattern. The notebook defines a bounded scientific function; Nemotron requests that function through a structured tool call; the notebook validates and executes it; the result is visualized immediately; and Nemotron provides a short interpretation. A final synthesis combines the numerical results from every stage into a detailed scientific discussion.

Brev supplies the GPU environment, nvMolKit performs the batched GPU chemistry operations, RDKit handles molecule parsing and display preparation, and the notebook enforces the execution and scientific-safety boundaries. Nemotron chooses validated tool parameters and explains results, but it does not execute arbitrary Python.

This is a cheminformatics demonstration, not a benchmark or validated scientific study. Fingerprints, similarities, clusters, force-field energies, and candidate geometries are computational outputs. They do not establish binding, biological activity, ADMET properties, efficacy, safety, synthesizability, or clinical relevance.

## 1. Preflight

Run this notebook in Brev-managed Jupyter on a compatible NVIDIA GPU. The hosted NVIDIA Developer API key is read from the environment when available or entered through a hidden notebook prompt; it is never displayed or stored by the notebook.

In [ ]:
import json
import os
import sys
from getpass import getpass
from pathlib import Path

PROJECT_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (
        (candidate / "data" / "sample_molecules.csv").is_file()
        and (candidate / "skills" / "nvmolkit" / "SKILL.md").is_file()
        and (candidate / "demo_agent.py").is_file()
    ):
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Run this notebook from the repository root or its notebooks/ directory."
    )

sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / "data" / "sample_molecules.csv"
SKILL_PATH = PROJECT_ROOT / "skills" / "nvmolkit" / "SKILL.md"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Markdown, display
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.rdDistGeom import ETKDGv3
from rdkit.Geometry import Point3D

from demo_agent import (
    ClusterArgs,
    ConformerArgs,
    FingerprintArgs,
    PrepareSampleArgs,
    ReadSkillArgs,
    SimilarityArgs,
    request_and_execute_step,
    request_brief_interpretation,
    request_final_synthesis,
)
import nvmolkit
from nvmolkit.clustering import fused_butina
from nvmolkit.embedMolecules import EmbedMolecules
from nvmolkit.fingerprints import MorganFingerprintGenerator
from nvmolkit.mmffOptimization import MMFFOptimizeMoleculesConfs
from nvmolkit.similarity import crossTanimotoSimilarity
from nvmolkit.types import CoordinateOutput

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-capable NVIDIA GPU is required.")
if torch.version.cuda is None:
    raise RuntimeError("The installed PyTorch build does not expose CUDA.")
cuda_capability = torch.cuda.get_device_capability(0)
if cuda_capability < (7, 0):
    raise RuntimeError("nvMolKit requires NVIDIA compute capability 7.0 or newer.")
if nvmolkit.__version__ != "0.5.0":
    raise RuntimeError(
        f"This notebook is pinned to nvMolKit 0.5.0; found {nvmolkit.__version__}."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA capability:", cuda_capability)
print("PyTorch/CUDA:", torch.__version__, torch.version.cuda)
print("nvMolKit:", nvmolkit.__version__)

probe_molecules = [
    Chem.MolFromSmiles(smiles) for smiles in ("CCO", "CCN", "c1ccccc1")
]
if any(molecule is None for molecule in probe_molecules):
    raise RuntimeError("The fixed GPU probe molecules did not parse.")
probe_fingerprints = MorganFingerprintGenerator(
    radius=2, fpSize=1024
).GetFingerprints(probe_molecules)
torch.cuda.synchronize()
if tuple(probe_fingerprints.torch().shape) != (3, 32):
    raise RuntimeError("The L4-compatible nvMolKit fingerprint probe failed.")
print("GPU fingerprint probe passed:", tuple(probe_fingerprints.torch().shape))

In [ ]:
api_key = os.environ.get("NVIDIA_API_KEY", "").strip()
if not api_key:
    api_key = getpass(
        "Hosted NVIDIA Developer API key from the Nemotron build.nvidia.com "
        "model page (starts with nvapi-; bare key only; input hidden): "
    ).strip()
    if not api_key:
        raise RuntimeError("A hosted NVIDIA Developer API key is required.")
model = "nvidia/nemotron-3-nano-30b-a3b"

## 2. Nemotron learns the nvMolKit skill

**Task.** Ask Nemotron to read the fixed, pinned BioNeMo Agent Toolkit skill before any sample analysis. The local executor exposes the five operations used by this notebook and the GPU-only boundary; no model-selected path is accepted.

In [ ]:
def read_nvmolkit_skill(args: ReadSkillArgs):
    del args
    skill_text = SKILL_PATH.read_text(encoding="utf-8")
    capabilities = [
        {
            "Capability": "Morgan fingerprints",
            "Entry point": "MorganFingerprintGenerator",
            "Role": "Molecular representation",
        },
        {
            "Capability": "Tanimoto similarity",
            "Entry point": "crossTanimotoSimilarity",
            "Role": "Pairwise structural similarity",
        },
        {
            "Capability": "Butina clustering",
            "Entry point": "fused_butina",
            "Role": "Structural grouping",
        },
        {
            "Capability": "ETKDG embedding",
            "Entry point": "EmbedMolecules",
            "Role": "Candidate 3D conformers",
        },
        {
            "Capability": "MMFF94 optimization",
            "Entry point": "MMFFOptimizeMoleculesConfs",
            "Role": "Force-field minimization",
        },
    ]
    summary = {
        "pinned_revision": "ce151c15470991c8cb9a0efdd531a124c346ca5b",
        "source": (
            "https://github.com/NVIDIA-BioNeMo/bionemo-agent-toolkit/blob/"
            "ce151c15470991c8cb9a0efdd531a124c346ca5b/"
            "library-skills/nvMolKit/SKILL.md"
        ),
        "gpu_boundary": (
            "An NVIDIA GPU with compute capability 7.0 or newer is required. "
            "There is no CPU fallback."
        ),
        "capabilities": capabilities,
        "figure_context": {
            "visual": "capability table",
            "rows": 5,
            "columns": ["Capability", "Entry point", "Role"],
        },
    }
    return {"skill_text": skill_text, "summary": summary}

In [ ]:
# Validation completes before the executor runs.
skill_decision, skill_artifact = request_and_execute_step(
    api_key,
    tool_name="read_nvmolkit_skill",
    task_prompt=(
        "Read the pinned nvMolKit skill and ground the guided chemistry workflow "
        "in its documented capabilities and GPU-only limitations."
    ),
    context={
        "artifact": "fixed vendored nvMolKit skill",
        "expected_capabilities": 5,
    },
    executor=read_nvmolkit_skill,
    model=model,
)
display(Markdown(f"**Requested tool:** `{skill_decision.tool_name}`"))
display(
    Markdown(
        "**Validated arguments:** `"
        + json.dumps(skill_decision.arguments.model_dump(mode="json"), sort_keys=True)
        + "`"
    )
)

In [ ]:
json.dumps(skill_artifact["summary"], allow_nan=False)
capability_table = pd.DataFrame(
    [
        {
            "Capability": "Morgan fingerprints",
            "Entry point": "MorganFingerprintGenerator",
            "Role": "Molecular representation",
        },
        {
            "Capability": "Tanimoto similarity",
            "Entry point": "crossTanimotoSimilarity",
            "Role": "Pairwise structural similarity",
        },
        {
            "Capability": "Butina clustering",
            "Entry point": "fused_butina",
            "Role": "Structural grouping",
        },
        {
            "Capability": "ETKDG embedding",
            "Entry point": "EmbedMolecules",
            "Role": "Candidate 3D conformers",
        },
        {
            "Capability": "MMFF94 optimization",
            "Entry point": "MMFFOptimizeMoleculesConfs",
            "Role": "Force-field minimization",
        },
    ]
)
display(capability_table)
skill_grounding = {
    "pinned_revision": skill_artifact["summary"]["pinned_revision"],
    "gpu_boundary": skill_artifact["summary"]["gpu_boundary"],
    "capabilities": [
        capability["Entry point"]
        for capability in skill_artifact["summary"]["capabilities"]
    ],
}

In [ ]:
try:
    skill_interpretation = request_brief_interpretation(
        api_key,
        skill_decision,
        {
            "summary": skill_artifact["summary"],
            "full_skill_text": skill_artifact["skill_text"],
        },
        {
            **skill_artifact["summary"]["figure_context"],
            "interpretation_scope": (
                "Explain the documented capabilities and GPU/API limitations."
            ),
        },
        model=model,
    )
except Exception:
    skill_interpretation = "Interpretation unavailable"
display(Markdown(skill_interpretation))

## 3. Molecular sample

**Task.** Ask Nemotron to validate and preview exactly 24 entries from the fixed bundled CSV. The executor checks the raw 256-row shape, excludes invalid SMILES while preserving metadata, and makes the exclusions visible.

In [ ]:
def prepare_molecular_sample(args: PrepareSampleArgs):
    # DATA_PATH is fixed; the model cannot choose a file or filesystem location.
    raw_sample = pd.read_csv(DATA_PATH)
    if (
        len(raw_sample) != 256
        or list(raw_sample.columns) != ["molecule_id", "smiles"]
    ):
        raise RuntimeError(
            "The bundled sample must contain exactly 256 rows and the expected columns."
        )

    parsed_molecules = [
        Chem.MolFromSmiles(str(smiles)) for smiles in raw_sample["smiles"]
    ]
    valid_mask = np.array(
        [molecule is not None for molecule in parsed_molecules], dtype=bool
    )
    # Invalid SMILES never enter GPU artifacts; their identifiers remain reportable.
    excluded_identifiers = [
        str(identifier)
        for identifier in raw_sample.loc[~valid_mask, "molecule_id"].tolist()
    ]
    molecules = [
        molecule for molecule in parsed_molecules if molecule is not None
    ]
    if not molecules:
        raise RuntimeError("The bundled sample produced zero valid molecules.")

    filtered_sample = raw_sample.loc[valid_mask].reset_index(drop=True).copy()
    summary = {
        "raw_rows": int(len(raw_sample)),
        "valid_molecules": int(len(molecules)),
        "invalid_molecules": int(len(excluded_identifiers)),
        "excluded_identifiers": excluded_identifiers,
        "preview_count": int(args.preview_count),
        "figure_context": {
            "visual": "2D molecule grid",
            "displayed_molecules": int(args.preview_count),
            "molecules_per_row": 6,
            "scope": "first valid molecules in bundled input order",
        },
    }
    return {
        "frame": filtered_sample,
        "molecules": molecules,
        "summary": summary,
    }

In [ ]:
# Validation completes before the executor runs.
sample_decision, sample_artifact = request_and_execute_step(
    api_key,
    tool_name="prepare_molecular_sample",
    task_prompt=(
        "Validate the fixed 256-row sample and preview exactly 24 valid molecules. "
        "Report all invalid SMILES exclusions."
    ),
    context={
        "skill_grounding": skill_grounding,
        "fixed_dataset": "data/sample_molecules.csv",
        "raw_rows_expected": 256,
        "preview_count": 24,
    },
    executor=prepare_molecular_sample,
    model=model,
)
display(Markdown(f"**Requested tool:** `{sample_decision.tool_name}`"))
display(
    Markdown(
        "**Validated arguments:** `"
        + json.dumps(sample_decision.arguments.model_dump(mode="json"), sort_keys=True)
        + "`"
    )
)

In [ ]:
json.dumps(sample_artifact["summary"], allow_nan=False)
excluded_text = (
    ", ".join(sample_artifact["summary"]["excluded_identifiers"])
    if sample_artifact["summary"]["excluded_identifiers"]
    else "none"
)
display(
    Markdown(
        f"**Invalid SMILES:** {sample_artifact['summary']['invalid_molecules']}; "
        f"**excluded identifiers:** {excluded_text}"
    )
)
display(
    pd.Series(
        {
            "Raw rows": sample_artifact["summary"]["raw_rows"],
            "Valid molecules": sample_artifact["summary"]["valid_molecules"],
            "Invalid molecules": sample_artifact["summary"]["invalid_molecules"],
            "Previewed molecules": sample_artifact["summary"]["preview_count"],
        },
        name="Sample",
    ).to_frame()
)
display(
    Draw.MolsToGridImage(
        sample_artifact["molecules"][:24],
        legends=sample_artifact["frame"]["molecule_id"].iloc[:24].tolist(),
        molsPerRow=6,
        subImgSize=(220, 180),
    )
)

In [ ]:
try:
    sample_interpretation = request_brief_interpretation(
        api_key,
        sample_decision,
        sample_artifact["summary"],
        {
            **sample_artifact["summary"]["figure_context"],
            "interpretation_scope": (
                "The 24-molecule preview cannot establish whole-library chemistry."
            ),
        },
        model=model,
    )
except Exception:
    sample_interpretation = "Interpretation unavailable"
display(Markdown(sample_interpretation))

## 4. Mapping molecular similarity

The following three guided calls build one representation, measure all-pairs similarity, and identify clusters. Each executor consumes only validated arguments plus local artifacts from earlier stages.

### 4.1 Morgan fingerprints

**Task.** Ask Nemotron to choose strict fingerprint parameters, recommending radius 2 and 1,024 bits. The executor computes packed Morgan fingerprints for every valid molecule and summarizes representation density without inferring biological activity.

In [ ]:
def compute_morgan_fingerprints(args: FingerprintArgs):
    generator = MorganFingerprintGenerator(
        radius=args.fingerprint_radius,
        fpSize=args.fingerprint_size,
    )
    fingerprints = generator.GetFingerprints(sample_artifact["molecules"])
    fingerprint_tensor = fingerprints.torch()
    expected_shape = (
        len(sample_artifact["molecules"]),
        args.fingerprint_size // 32,
    )
    if tuple(fingerprint_tensor.shape) != expected_shape:
        raise RuntimeError("The packed Morgan fingerprint tensor shape was unexpected.")

    # Each GPU-resident int32 word packs 32 hashed fingerprint bits.
    packed_words = fingerprint_tensor.to(torch.int64) & 0xFFFFFFFF
    bit_positions = torch.arange(
        32, dtype=torch.int64, device=fingerprint_tensor.device
    )
    active_bits_gpu = (
        (packed_words.unsqueeze(-1) >> bit_positions) & 1
    ).sum(dim=(1, 2))
    # Synchronize GPU work before moving active hashed bits into a host summary.
    torch.cuda.synchronize()
    active_bits = active_bits_gpu.cpu().numpy().astype(np.int64)

    summary = {
        "tensor_shape": [int(value) for value in fingerprint_tensor.shape],
        "radius": int(args.fingerprint_radius),
        "size": int(args.fingerprint_size),
        "molecule_count": int(len(sample_artifact["molecules"])),
        "device": str(fingerprint_tensor.device),
        "min_active_bits": int(active_bits.min()),
        "median_active_bits": float(np.median(active_bits)),
        "mean_active_bits": float(np.mean(active_bits)),
        "max_active_bits": int(active_bits.max()),
        "figure_context": {
            "visual": "histogram",
            "x_axis": "active hashed bits per molecule",
            "y_axis": "molecule count",
            "bins": 20,
        },
    }
    return {
        "fingerprints": fingerprints,
        "tensor": fingerprint_tensor,
        "active_bits": active_bits,
        "summary": summary,
    }

In [ ]:
# Validation completes before the executor runs.
fingerprint_decision, fingerprint_artifact = request_and_execute_step(
    api_key,
    tool_name="compute_morgan_fingerprints",
    task_prompt=(
        "Generate Morgan fingerprints for all valid molecules. Use the recommended "
        "radius 2 and 1024-bit representation unless a different allowed value is "
        "scientifically justified by the bounded context."
    ),
    context={
        "skill_grounding": skill_grounding,
        "sample_summary": sample_artifact["summary"],
        "recommended": {"fingerprint_radius": 2, "fingerprint_size": 1024},
    },
    executor=compute_morgan_fingerprints,
    model=model,
)
display(Markdown(f"**Requested tool:** `{fingerprint_decision.tool_name}`"))
display(
    Markdown(
        "**Validated arguments:** `"
        + json.dumps(
            fingerprint_decision.arguments.model_dump(mode="json"), sort_keys=True
        )
        + "`"
    )
)

In [ ]:
json.dumps(fingerprint_artifact["summary"], allow_nan=False)
display(
    pd.Series(
        {
            "Tensor shape": fingerprint_artifact["summary"]["tensor_shape"],
            "Radius": fingerprint_artifact["summary"]["radius"],
            "Fingerprint bits": fingerprint_artifact["summary"]["size"],
            "Minimum active bits": fingerprint_artifact["summary"]["min_active_bits"],
            "Median active bits": fingerprint_artifact["summary"]["median_active_bits"],
            "Mean active bits": fingerprint_artifact["summary"]["mean_active_bits"],
            "Maximum active bits": fingerprint_artifact["summary"]["max_active_bits"],
        },
        name="Morgan fingerprints",
    ).to_frame()
)
plt.figure(figsize=(7, 4))
plt.hist(
    fingerprint_artifact["active_bits"],
    bins=20,
    color="#76B900",
    edgecolor="black",
)
plt.title("Active Morgan fingerprint bits per molecule")
plt.xlabel("Active hashed bits")
plt.ylabel("Molecule count")
plt.tight_layout()
plt.show()

In [ ]:
try:
    fingerprint_interpretation = request_brief_interpretation(
        api_key,
        fingerprint_decision,
        fingerprint_artifact["summary"],
        {
            **fingerprint_artifact["summary"]["figure_context"],
            "interpretation_scope": (
                "Interpret representation and density only; do not infer biological activity."
            ),
        },
        model=model,
    )
except Exception:
    fingerprint_interpretation = "Interpretation unavailable"
display(Markdown(fingerprint_interpretation))

### 4.2 All-pairs Tanimoto similarity

**Task.** Ask Nemotron to request the argument-free similarity stage. The executor reuses the local GPU fingerprint artifact, validates the square 0–1 matrix, excludes trivial diagonal self-similarity from statistics, and reports the most similar off-diagonal molecule pair.

In [ ]:
def compute_tanimoto_similarity(args: SimilarityArgs):
    del args
    similarity_result = crossTanimotoSimilarity(
        fingerprint_artifact["fingerprints"]
    )
    torch.cuda.synchronize()
    similarity_matrix = similarity_result.torch().cpu().numpy()
    molecule_count = len(sample_artifact["molecules"])

    if similarity_matrix.shape != (molecule_count, molecule_count):
        raise RuntimeError("The all-pairs Tanimoto matrix shape was unexpected.")
    if not np.isfinite(similarity_matrix).all():
        raise RuntimeError("The Tanimoto matrix contains non-finite values.")
    if not np.allclose(
        similarity_matrix, similarity_matrix.T, rtol=0, atol=1e-7
    ):
        raise RuntimeError("The Tanimoto matrix is not symmetric.")
    if np.any((similarity_matrix < 0) | (similarity_matrix > 1)):
        raise RuntimeError("Tanimoto values must remain on the 0-1 scale.")

    # Self-similarity on the diagonal is trivially 1.0, so exclude it from statistics.
    upper_rows, upper_columns = np.triu_indices(molecule_count, k=1)
    off_diagonal = similarity_matrix[upper_rows, upper_columns]
    pair_position = int(np.argmax(off_diagonal))
    first_index = int(upper_rows[pair_position])
    second_index = int(upper_columns[pair_position])
    molecule_ids = sample_artifact["frame"]["molecule_id"].astype(str).tolist()

    summary = {
        "shape": [int(value) for value in similarity_matrix.shape],
        "q1": float(np.quantile(off_diagonal, 0.25)),
        "median": float(np.median(off_diagonal)),
        "q3": float(np.quantile(off_diagonal, 0.75)),
        "p90": float(np.quantile(off_diagonal, 0.90)),
        "max_off_diagonal": float(off_diagonal[pair_position]),
        "most_similar_nonidentical_pair_ids": [
            molecule_ids[first_index],
            molecule_ids[second_index],
        ],
        "figure_context": {
            "visual": "unordered heatmap",
            "x_axis": "molecules in validated input order",
            "y_axis": "molecules in validated input order",
            "scale": [0.0, 1.0],
            "color": "Tanimoto similarity",
        },
    }
    return {
        "result": similarity_result,
        "matrix": similarity_matrix,
        "summary": summary,
    }

In [ ]:
# Validation completes before the executor runs.
similarity_decision, similarity_artifact = request_and_execute_step(
    api_key,
    tool_name="compute_tanimoto_similarity",
    task_prompt=(
        "Measure all-pairs Tanimoto similarity from the existing local fingerprint "
        "artifact without selecting any additional parameters."
    ),
    context={
        "skill_grounding": skill_grounding,
        "fingerprint_summary": fingerprint_artifact["summary"],
    },
    executor=compute_tanimoto_similarity,
    model=model,
)
display(Markdown(f"**Requested tool:** `{similarity_decision.tool_name}`"))
display(
    Markdown(
        "**Validated arguments:** `"
        + json.dumps(similarity_decision.arguments.model_dump(mode="json"), sort_keys=True)
        + "`"
    )
)

In [ ]:
json.dumps(similarity_artifact["summary"], allow_nan=False)
display(
    pd.Series(
        {
            "Matrix shape": similarity_artifact["summary"]["shape"],
            "Q1 (off-diagonal)": similarity_artifact["summary"]["q1"],
            "Median (off-diagonal)": similarity_artifact["summary"]["median"],
            "Q3 (off-diagonal)": similarity_artifact["summary"]["q3"],
            "90th percentile": similarity_artifact["summary"]["p90"],
            "Maximum off-diagonal": similarity_artifact["summary"]["max_off_diagonal"],
            "Most similar nonidentical pair": " / ".join(
                similarity_artifact["summary"][
                    "most_similar_nonidentical_pair_ids"
                ]
            ),
        },
        name="Tanimoto similarity",
    ).to_frame()
)
plt.figure(figsize=(8, 7))
sns.heatmap(
    similarity_artifact["matrix"],
    cmap="viridis",
    vmin=0,
    vmax=1,
    cbar_kws={"label": "Tanimoto similarity"},
)
plt.title("Unordered all-pairs Tanimoto similarity (validated input order)")
plt.xlabel("Molecule index in input order")
plt.ylabel("Molecule index in input order")
plt.tight_layout()
plt.show()

In [ ]:
try:
    similarity_interpretation = request_brief_interpretation(
        api_key,
        similarity_decision,
        similarity_artifact["summary"],
        {
            **similarity_artifact["summary"]["figure_context"],
            "interpretation_scope": (
                "Explain the off-diagonal distribution and most-similar pair without "
                "inferring shared biological activity."
            ),
        },
        model=model,
    )
except Exception:
    similarity_interpretation = "Interpretation unavailable"
display(Markdown(similarity_interpretation))

### 4.3 Fused Butina clusters

**Task.** Ask Nemotron to cluster the existing fingerprints, recommending a 0.50 Tanimoto-distance cutoff within the strict 0.40–0.60 range. The executor validates complete, unique assignment and reports singletons explicitly because cluster fragmentation depends on the selected cutoff.

In [ ]:
def cluster_with_fused_butina(args: ClusterArgs):
    fingerprints = fingerprint_artifact["fingerprints"]
    cutoff = float(args.cluster_cutoff)
    # The cutoff is a Tanimoto-distance threshold: similarity > 1 - cutoff.
    # Lowering it requires greater similarity and can create more singletons.
    clusters, reported_cluster_sizes = fused_butina(
        fingerprints.torch(), cutoff=cutoff
    )
    torch.cuda.synchronize()

    molecule_count = len(sample_artifact["molecules"])
    assigned_indices = [
        int(molecule_index)
        for cluster in clusters
        for molecule_index in cluster
    ]
    if (
        len(assigned_indices) != molecule_count
        or sorted(assigned_indices) != list(range(molecule_count))
    ):
        raise RuntimeError("Every molecule must be assigned exactly once.")

    assignments = np.full(molecule_count, -1, dtype=int)
    for cluster_id, cluster in enumerate(clusters):
        for molecule_index in cluster:
            assignments[int(molecule_index)] = int(cluster_id)
    cluster_sizes = [int(len(cluster)) for cluster in clusters]
    singleton_count = int(sum(size == 1 for size in cluster_sizes))
    largest_cluster_sizes = sorted(cluster_sizes, reverse=True)[:15]

    summary = {
        "cutoff": cutoff,
        "cluster_count": int(len(clusters)),
        "singleton_count": singleton_count,
        "singleton_fraction": float(singleton_count / molecule_count),
        "largest_cluster_sizes": largest_cluster_sizes,
        "molecule_count": int(molecule_count),
        "figure_context": {
            "visual": "bar chart of 15 largest clusters",
            "x_axis": "cluster rank by descending size",
            "y_axis": "molecule count",
            "singleton_count": singleton_count,
            "cutoff": cutoff,
        },
    }
    return {
        "assignments": assignments,
        "clusters": clusters,
        "reported_cluster_sizes": reported_cluster_sizes,
        "summary": summary,
    }

In [ ]:
# Validation completes before the executor runs.
cluster_decision, cluster_artifact = request_and_execute_step(
    api_key,
    tool_name="cluster_with_fused_butina",
    task_prompt=(
        "Cluster the existing Morgan fingerprints with fused Butina. Use the "
        "recommended Tanimoto-distance cutoff 0.50 unless another allowed value is justified."
    ),
    context={
        "skill_grounding": skill_grounding,
        "fingerprint_summary": fingerprint_artifact["summary"],
        "similarity_summary": similarity_artifact["summary"],
        "recommended": {"cluster_cutoff": 0.50},
    },
    executor=cluster_with_fused_butina,
    model=model,
)
display(Markdown(f"**Requested tool:** `{cluster_decision.tool_name}`"))
display(
    Markdown(
        "**Validated arguments:** `"
        + json.dumps(cluster_decision.arguments.model_dump(mode="json"), sort_keys=True)
        + "`"
    )
)

In [ ]:
json.dumps(cluster_artifact["summary"], allow_nan=False)
display(
    pd.Series(
        {
            "Cutoff": cluster_artifact["summary"]["cutoff"],
            "Clusters": cluster_artifact["summary"]["cluster_count"],
            "Singletons": cluster_artifact["summary"]["singleton_count"],
            "Singleton fraction": cluster_artifact["summary"]["singleton_fraction"],
            "Molecules assigned": cluster_artifact["summary"]["molecule_count"],
        },
        name="Fused Butina clustering",
    ).to_frame()
)
top_cluster_sizes = sorted(
    [len(cluster) for cluster in cluster_artifact["clusters"]], reverse=True
)[:15]
cluster_ranks = np.arange(1, len(top_cluster_sizes) + 1)
plt.figure(figsize=(8, 4))
plt.bar(cluster_ranks, top_cluster_sizes, color="#76B900")
plt.title(
    "15 largest fused Butina clusters "
    f"(singletons: {cluster_artifact['summary']['singleton_count']})"
)
plt.xlabel("Cluster rank by descending size")
plt.ylabel("Molecule count")
plt.xticks(cluster_ranks)
plt.tight_layout()
plt.show()

In [ ]:
try:
    cluster_interpretation = request_brief_interpretation(
        api_key,
        cluster_decision,
        cluster_artifact["summary"],
        {
            **cluster_artifact["summary"]["figure_context"],
            "interpretation_scope": (
                "Discuss fragmentation, diversity, singletons, and cutoff sensitivity "
                "without biological claims."
            ),
        },
        model=model,
    )
except Exception:
    cluster_interpretation = "Interpretation unavailable"
display(Markdown(cluster_interpretation))

## 5. Conformers and MMFF94

**Task.** Select MMFF94-eligible representatives from distinct clusters, recommending 4 representatives and 4 conformers per representative. Sample candidates with ETKDGv3, minimize them with nvMolKit MMFF94, and compare energies only within each molecule.

In [ ]:
def generate_and_optimize_conformers(args: ConformerArgs):
    molecule_ids = sample_artifact["frame"]["molecule_id"].astype(str).tolist()
    cluster_assignments = cluster_artifact["assignments"]
    eligible_candidates = []
    for molecule_index, molecule in enumerate(sample_artifact["molecules"]):
        hydrogenated = Chem.AddHs(Chem.Mol(molecule))
        if AllChem.MMFFHasAllMoleculeParams(hydrogenated):
            eligible_candidates.append(
                {
                    "representative_id": molecule_ids[molecule_index],
                    "molecule_index": int(molecule_index),
                    "cluster_id": int(cluster_assignments[molecule_index]),
                    "heavy_atom_count": int(molecule.GetNumHeavyAtoms()),
                    "molecule": hydrogenated,
                }
            )

    # Representatives are chosen from distinct clusters by lower heavy-atom count, then stable molecule index.
    eligible_candidates = sorted(
        eligible_candidates,
        key=lambda candidate: (
            candidate["heavy_atom_count"],
            candidate["molecule_index"],
        ),
    )
    eligible_distinct_cluster_count = len(
        {candidate["cluster_id"] for candidate in eligible_candidates}
    )
    representatives = []
    selected_cluster_ids = set()
    for candidate in eligible_candidates:
        if candidate["cluster_id"] in selected_cluster_ids:
            continue
        representatives.append(candidate)
        selected_cluster_ids.add(candidate["cluster_id"])
        if len(representatives) == args.representative_count:
            break
    if not representatives:
        raise RuntimeError(
            "At least one MMFF94-eligible representative from a distinct cluster is required."
        )
    selection_notice = None
    if len(representatives) < args.representative_count:
        selection_notice = (
            "Selected fewer distinct eligible representatives than requested: "
            f"{len(representatives)} of {args.representative_count}."
        )

    conformer_molecules = [representative["molecule"] for representative in representatives]
    embedding_parameters = AllChem.ETKDGv3()
    embedding_parameters.useRandomCoords = True
    embedding_parameters.randomSeed = 7
    EmbedMolecules(
        conformer_molecules,
        embedding_parameters,
        confsPerMolecule=args.conformers_per_representative,
        maxIterations=-1,
    )

    # Zero embeddings are recorded and excluded; partial embeddings continue to MMFF94.
    zero_embedding_representatives = []
    partial_embedding_representatives = []
    successful_representatives = []
    for representative, molecule in zip(representatives, conformer_molecules):
        generated_count = int(molecule.GetNumConformers())
        detail = {
            "representative_id": representative["representative_id"],
            "molecule_index": representative["molecule_index"],
            "cluster_id": representative["cluster_id"],
            "generated_conformer_count": generated_count,
            "requested_conformer_count": int(args.conformers_per_representative),
        }
        if generated_count == 0:
            zero_embedding_representatives.append(detail)
        else:
            successful_representatives.append({**representative, **detail})
            if generated_count < args.conformers_per_representative:
                partial_embedding_representatives.append(detail)
    if not successful_representatives:
        raise RuntimeError(
            "All representatives produced zero conformers; MMFF94 cannot run."
        )

    mmff_molecules = [
        representative["molecule"] for representative in successful_representatives
    ]
    optimization_result = MMFFOptimizeMoleculesConfs(
        mmff_molecules,
        maxIters=500,
        output=CoordinateOutput.DEVICE,
    )
    torch.cuda.synchronize()
    energies = optimization_result.energies.torch().detach().cpu()
    convergence = optimization_result.converged.torch().detach().cpu()
    mol_indices = optimization_result.mol_indices.torch().detach().cpu()
    conf_indices = optimization_result.conf_indices.torch().detach().cpu()
    per_molecule = optimization_result.per_molecule()

    generated_conformer_count = sum(
        molecule.GetNumConformers() for molecule in mmff_molecules
    )
    if not (
        len(energies)
        == len(convergence)
        == len(mol_indices)
        == len(conf_indices)
        == generated_conformer_count
    ):
        raise RuntimeError("MMFF94 result buffers do not cover every generated conformer.")
    convergence_values = [int(value) for value in convergence.tolist()]
    if not set(convergence_values) <= {0, 1}:
        raise RuntimeError("MMFF94 convergence flags must be binary.")
    result_pairs = [
        (int(molecule_index), int(conformer_index))
        for molecule_index, conformer_index in zip(
            mol_indices.tolist(), conf_indices.tolist()
        )
    ]
    expected_pairs = {
        (molecule_index, conformer_index)
        for molecule_index, molecule in enumerate(mmff_molecules)
        for conformer_index in range(molecule.GetNumConformers())
    }
    if len(set(result_pairs)) != len(result_pairs) or set(result_pairs) != expected_pairs:
        raise RuntimeError("MMFF94 molecule/conformer indices are incomplete or duplicated.")
    if len(per_molecule) != len(mmff_molecules):
        raise RuntimeError("MMFF94 per-molecule coordinates do not match the input batch.")

    # Optimized device coordinates are copied back into their matching RDKit conformers for reliable static rendering.
    coordinate_offsets = [0 for _ in mmff_molecules]
    for molecule_index, conformer_index in result_pairs:
        conformer_coordinates = per_molecule[molecule_index]
        coordinate_offset = coordinate_offsets[molecule_index]
        if coordinate_offset >= len(conformer_coordinates):
            raise RuntimeError(
                "MMFF94 grouped coordinates do not cover each flat result pair."
            )
        device_coordinates = conformer_coordinates[coordinate_offset]
        coordinate_offsets[molecule_index] += 1
        molecule = mmff_molecules[molecule_index]
        coordinates = device_coordinates.detach().cpu().numpy()
        if coordinates.shape != (molecule.GetNumAtoms(), 3):
            raise RuntimeError("An optimized coordinate array has the wrong shape.")
        if not np.isfinite(coordinates).all():
            raise RuntimeError("Optimized coordinates contain non-finite values.")
        conformer = molecule.GetConformer(conformer_index)
        for atom_index, (x_coordinate, y_coordinate, z_coordinate) in enumerate(
            coordinates
        ):
            conformer.SetAtomPosition(
                atom_index,
                Point3D(
                    float(x_coordinate),
                    float(y_coordinate),
                    float(z_coordinate),
                ),
            )
    if coordinate_offsets != [
        len(conformer_coordinates) for conformer_coordinates in per_molecule
    ]:
        raise RuntimeError("MMFF94 grouped coordinates were not fully consumed.")

    per_conformer_records = []
    for energy, did_converge, molecule_index, conformer_index in zip(
        energies.tolist(),
        convergence_values,
        mol_indices.tolist(),
        conf_indices.tolist(),
    ):
        representative = successful_representatives[int(molecule_index)]
        numeric_energy = float(energy)
        per_conformer_records.append(
            {
                "representative_id": representative["representative_id"],
                "molecule_index": representative["molecule_index"],
                "cluster_id": representative["cluster_id"],
                "optimization_molecule_index": int(molecule_index),
                "conformer_index": int(conformer_index),
                "energy_kcal_mol": (
                    numeric_energy if np.isfinite(numeric_energy) else None
                ),
                "converged": bool(did_converge),
            }
        )
    per_conformer_records.sort(
        key=lambda record: (
            record["optimization_molecule_index"],
            record["conformer_index"],
        )
    )

    # Raw MMFF energies are comparable only within one molecule; molecules are never ranked against each other.
    selected_conformer_records = []
    for molecule_index, representative in enumerate(successful_representatives):
        converged_records = [
            record
            for record in per_conformer_records
            if record["optimization_molecule_index"] == molecule_index
            and record["converged"]
            and record["energy_kcal_mol"] is not None
        ]
        if converged_records:
            selected = min(
                converged_records,
                key=lambda record: (
                    record["energy_kcal_mol"], record["conformer_index"]
                ),
            ).copy()
            selected["selected_conformer_id"] = (
                f"{representative['representative_id']}:conf-{selected['conformer_index']}"
            )
            selected_conformer_records.append(selected)

    representative_identifiers = [
        {
            "representative_id": representative["representative_id"],
            "molecule_index": representative["molecule_index"],
            "cluster_id": representative["cluster_id"],
            "heavy_atom_count": representative["heavy_atom_count"],
        }
        for representative in representatives
    ]
    attempted_conformer_count = len(per_conformer_records)
    converged_conformer_count = sum(
        record["converged"] for record in per_conformer_records
    )
    summary = {
        "requested_representative_count": int(args.representative_count),
        "requested_conformers_per_representative": int(
            args.conformers_per_representative
        ),
        "requested_conformer_count": int(
            args.representative_count * args.conformers_per_representative
        ),
        "mmff94_eligible_molecule_count": int(len(eligible_candidates)),
        "eligible_distinct_cluster_count": int(eligible_distinct_cluster_count),
        "selected_representative_count": int(len(representatives)),
        "successful_embedding_representative_count": int(
            len(successful_representatives)
        ),
        "selection_notice": selection_notice,
        "representative_identifiers": representative_identifiers,
        "generated_conformer_count": int(generated_conformer_count),
        "attempted_conformer_count": int(attempted_conformer_count),
        "converged_conformer_count": int(converged_conformer_count),
        "unconverged_conformer_count": int(
            attempted_conformer_count - converged_conformer_count
        ),
        "zero_embedding_representatives": zero_embedding_representatives,
        "partial_embedding_representatives": partial_embedding_representatives,
        "per_conformer_records": per_conformer_records,
        "selected_conformer_records": selected_conformer_records,
        "selected_conformer_ids": [
            record["selected_conformer_id"]
            for record in selected_conformer_records
        ],
        "figure_context": {
            "conformer_energy_plot": {
                "visual": "per-molecule scatter plot of every attempted conformer",
                "y_axis": "MMFF94 energy (kcal/mol)",
                "markers": "converged, unconverged, and missing/non-finite energy",
                "scope": "energies are ranked only within each molecule",
                "attempted_conformers": int(attempted_conformer_count),
            },
            "lowest_energy_conformers": {
                "visual": "static Matplotlib 3D atom-and-bond panels",
                "selection": "lowest-energy converged conformer within each molecule",
                "displayed_representatives": int(
                    min(6, len(selected_conformer_records))
                ),
                "scope": "sampled MMFF94 minima, not global or experimental conformations",
            },
        },
    }
    json.dumps(summary, allow_nan=False)
    return {
        "optimized_molecules": mmff_molecules,
        "energies": energies,
        "convergence": convergence,
        "mol_indices": mol_indices,
        "conf_indices": conf_indices,
        "per_molecule": per_molecule,
        "summary": summary,
    }


def plot_conformer_energies(conformer_artifact):
    records = conformer_artifact["summary"]["per_conformer_records"]
    representative_ids = list(
        dict.fromkeys(record["representative_id"] for record in records)
    )
    finite_energies = [
        record["energy_kcal_mol"]
        for record in records
        if record["energy_kcal_mol"] is not None
    ]
    if finite_energies:
        energy_span = max(finite_energies) - min(finite_energies)
        missing_energy_level = min(finite_energies) - max(1.0, 0.12 * energy_span)
    else:
        missing_energy_level = 0.0

    figure, axis = plt.subplots(figsize=(9, 4.8))
    labels_used = set()
    for record in records:
        molecule_position = representative_ids.index(record["representative_id"])
        horizontal_position = molecule_position + 0.06 * (record["conformer_index"] - 1.5)
        if record["energy_kcal_mol"] is None:
            marker, color, label, vertical_position = "v", "#7f7f7f", "Missing/non-finite energy", missing_energy_level
        elif record["converged"]:
            marker, color, label, vertical_position = "o", "#76B900", "Converged", record["energy_kcal_mol"]
        else:
            marker, color, label, vertical_position = "x", "#d62728", "Unconverged", record["energy_kcal_mol"]
        axis.scatter(
            horizontal_position,
            vertical_position,
            marker=marker,
            color=color,
            s=55,
            label=label if label not in labels_used else None,
        )
        labels_used.add(label)
        if record["energy_kcal_mol"] is None:
            axis.annotate("missing", (horizontal_position, vertical_position), fontsize=7)
    axis.set_xticks(range(len(representative_ids)), representative_ids, rotation=20)
    axis.set_ylabel("MMFF94 energy (kcal/mol)")
    axis.set_xlabel("Representative molecule (comparisons within molecule only)")
    axis.set_title("Conformer convergence and within molecule energy ranking")
    axis.legend(loc="best")
    axis.grid(axis="y", alpha=0.25)
    figure.tight_layout()
    plt.show()
    return figure


def plot_lowest_energy_conformers(conformer_artifact):
    selected_records = conformer_artifact["summary"]["selected_conformer_records"][:6]
    if not selected_records:
        figure, axis = plt.subplots(figsize=(8, 2.8))
        axis.axis("off")
        axis.text(
            0.5,
            0.5,
            "No conformer converged with a finite MMFF94 energy.",
            ha="center",
            va="center",
        )
        figure.tight_layout()
        plt.show()
        return figure

    panel_columns = min(2, len(selected_records))
    panel_rows = int(np.ceil(len(selected_records) / panel_columns))
    figure = plt.figure(figsize=(6.5 * panel_columns, 5.2 * panel_rows))
    element_colors = {1: "#d9d9d9", 6: "#4d4d4d", 7: "#377eb8", 8: "#e41a1c", 9: "#4daf4a", 15: "#ff7f00", 16: "#ffd92f", 17: "#4daf4a"}
    for panel_index, record in enumerate(selected_records, start=1):
        axis = figure.add_subplot(panel_rows, panel_columns, panel_index, projection="3d")
        molecule = conformer_artifact["optimized_molecules"][
            record["optimization_molecule_index"]
        ]
        conformer = molecule.GetConformer(record["conformer_index"])
        atom_coordinates = np.array(
            [
                list(conformer.GetAtomPosition(atom_index))
                for atom_index in range(molecule.GetNumAtoms())
            ],
            dtype=float,
        )
        atom_colors = [
            element_colors.get(atom.GetAtomicNum(), "#984ea3")
            for atom in molecule.GetAtoms()
        ]
        axis.scatter(
            atom_coordinates[:, 0],
            atom_coordinates[:, 1],
            atom_coordinates[:, 2],
            c=atom_colors,
            s=55,
            depthshade=True,
        )
        for bond in molecule.GetBonds():
            begin = bond.GetBeginAtomIdx()
            end = bond.GetEndAtomIdx()
            axis.plot(
                atom_coordinates[[begin, end], 0],
                atom_coordinates[[begin, end], 1],
                atom_coordinates[[begin, end], 2],
                color="#777777",
                linewidth=1.5,
            )
        axis.set_title(
            f"{record['representative_id']} | conf {record['conformer_index']}\n"
            f"within-molecule MMFF94: {record['energy_kcal_mol']:.3f} kcal/mol"
        )
        axis.set_axis_off()
        coordinate_range = np.ptp(atom_coordinates, axis=0)
        axis.set_box_aspect(np.maximum(coordinate_range, 1e-6))
    figure.suptitle("Lowest-energy converged sampled conformer per representative")
    figure.tight_layout()
    plt.show()
    return figure

In [ ]:
# Validation completes before the executor runs.
conformer_decision, conformer_artifact = request_and_execute_step(
    api_key,
    tool_name="generate_and_optimize_conformers",
    task_prompt=(
        "Select MMFF94-eligible representatives from distinct clusters and optimize "
        "ETKDGv3 conformers. Use the recommended 4 representatives and 4 conformers "
        "per representative unless another allowed value is justified."
    ),
    context={
        "skill_grounding": skill_grounding,
        "sample_summary": sample_artifact["summary"],
        "cluster_summary": cluster_artifact["summary"],
        "recommended": {
            "representative_count": 4,
            "conformers_per_representative": 4,
        },
    },
    executor=generate_and_optimize_conformers,
    model=model,
)
display(Markdown(f"**Requested tool:** `{conformer_decision.tool_name}`"))
display(
    Markdown(
        "**Validated arguments:** `"
        + json.dumps(
            conformer_decision.arguments.model_dump(mode="json"), sort_keys=True
        )
        + "`"
    )
)

In [ ]:
json.dumps(conformer_artifact["summary"], allow_nan=False)
display(
    pd.Series(
        {
            "Representatives selected": conformer_artifact["summary"]["selected_representative_count"],
            "Conformers requested": conformer_artifact["summary"]["requested_conformer_count"],
            "Conformers generated/attempted": conformer_artifact["summary"]["attempted_conformer_count"],
            "Converged": conformer_artifact["summary"]["converged_conformer_count"],
            "Unconverged": conformer_artifact["summary"]["unconverged_conformer_count"],
            "Partial embeddings": len(conformer_artifact["summary"]["partial_embedding_representatives"]),
            "Zero embeddings": len(conformer_artifact["summary"]["zero_embedding_representatives"]),
        },
        name="ETKDGv3 + MMFF94",
    ).to_frame()
)
conformer_result_table = pd.DataFrame(
    conformer_artifact["summary"]["per_conformer_records"]
).rename(
    columns={
        "representative_id": "Representative",
        "cluster_id": "Cluster",
        "conformer_index": "Conformer",
        "energy_kcal_mol": "MMFF94 energy (kcal/mol)",
        "converged": "Converged",
    }
)
display(
    conformer_result_table[
        ["Representative", "Cluster", "Conformer", "MMFF94 energy (kcal/mol)", "Converged"]
    ]
)
# Static Matplotlib figures remain authoritative and render before the optional py3Dmol enhancement.
plot_conformer_energies(conformer_artifact)
plot_lowest_energy_conformers(conformer_artifact)

In [ ]:
try:
    import py3Dmol

    selected_conformer_records = conformer_artifact["summary"]["selected_conformer_records"]
    if not selected_conformer_records:
        raise RuntimeError("No converged conformer is available for an interactive view.")
    for selected_record in selected_conformer_records:
        selected_molecule = conformer_artifact["optimized_molecules"][
            selected_record["optimization_molecule_index"]
        ]
        viewer = py3Dmol.view(width=520, height=360)
        viewer.addModel(
            Chem.MolToMolBlock(
                selected_molecule,
                confId=selected_record["conformer_index"],
            ),
            "mol",
        )
        viewer.setStyle({"stick": {}, "sphere": {"scale": 0.28}})
        viewer.zoomTo()
        viewer.show()
except Exception:
    display("Optional interactive 3D view unavailable.")

In [ ]:
try:
    conformer_interpretation = request_brief_interpretation(
        api_key,
        conformer_decision,
        conformer_artifact["summary"],
        {
            **conformer_artifact["summary"]["figure_context"],
            "interpretation_scope": (
                "Discuss convergence, sampling, and within-molecule energy ranking only."
            ),
        },
        model=model,
    )
except Exception:
    display("Interpretation unavailable")
else:
    display(Markdown(conformer_interpretation))

## 6. What the results mean

The actual stage summaries are passed to Nemotron for a detailed, PhD-level but presentation-readable interpretation. The model is text-only and receives textual figure descriptions, not pixels.

In [ ]:
analysis_summary = {
    "skill": skill_artifact["summary"],
    "sample": sample_artifact["summary"],
    "fingerprints": fingerprint_artifact["summary"],
    "similarity": similarity_artifact["summary"],
    "clusters": cluster_artifact["summary"],
    "conformers_mmff94": conformer_artifact["summary"],
}
serialized_analysis_summary = json.dumps(analysis_summary, allow_nan=False)
presentation_summary = pd.DataFrame(
    [
        {
            "stage": "Skill grounding",
            "key quantitative result": f"{len(skill_artifact['summary']['capabilities'])} documented capabilities",
        },
        {
            "stage": "Molecular sample",
            "key quantitative result": (
                f"{sample_artifact['summary']['valid_molecules']} valid; "
                f"{sample_artifact['summary']['invalid_molecules']} excluded"
            ),
        },
        {
            "stage": "Morgan fingerprints",
            "key quantitative result": (
                f"{fingerprint_artifact['summary']['size']} bits; mean active "
                f"{fingerprint_artifact['summary']['mean_active_bits']:.1f}"
            ),
        },
        {
            "stage": "Tanimoto similarity",
            "key quantitative result": (
                f"median {similarity_artifact['summary']['median']:.3f}; max off-diagonal "
                f"{similarity_artifact['summary']['max_off_diagonal']:.3f}"
            ),
        },
        {
            "stage": "Fused Butina clusters",
            "key quantitative result": (
                f"{cluster_artifact['summary']['cluster_count']} clusters; "
                f"{cluster_artifact['summary']['singleton_count']} singletons"
            ),
        },
        {
            "stage": "ETKDGv3 + MMFF94",
            "key quantitative result": (
                f"{conformer_artifact['summary']['converged_conformer_count']}/"
                f"{conformer_artifact['summary']['attempted_conformer_count']} converged"
            ),
        },
    ]
)
display(presentation_summary)

In [ ]:
scientific_boundary = (
    "These computational results do not establish binding, biological activity, "
    "ADMET, efficacy, safety, synthesizability, clinical relevance, or "
    "experimentally validated conformations. Sampled force-field minima are not "
    "global or experimental conformations."
)
display(scientific_boundary)

In [ ]:
try:
    final_synthesis = request_final_synthesis(
        api_key, analysis_summary, model=model
    )
except Exception:
    display("Final synthesis unavailable")
else:
    display(Markdown(final_synthesis))

In [ ]:
display(scientific_boundary)